In [5]:
import sys
!"{sys.executable}" -m pip install pyinstaller

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 2.8 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 4.5 MB/s eta 0:00:00

   ------------------------ --------------- 3/5 [pyinstaller-hooks-contrib]
   ------------------------ --------------- 3/5 [pyinstaller-hooks-contrib]
   ------------------------ --------------- 3/5 [pyinstaller-hooks-contrib]
   ------------------------ --------------- 3/5 [pyinstaller-hooks-contrib]
   -------------------------------- ------- 4/5 [pyinstaller]
   -------------------------------- ------- 4/5 [pyinstaller]
   -------------------------------- ------- 4/5 [pyinstaller]
   -------------------------------- ------- 4/5 [pyinstaller]
   -------------------------------- ------- 4/5 [pyinstaller]
   ---------------------------------------- 5/5 [pyinstaller]




[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: C:\Program Files\Python39\python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

ARTIFACT_DIR = "model"

imputer = joblib.load(os.path.join(ARTIFACT_DIR, "imputer.joblib"))
scaler = joblib.load(os.path.join(ARTIFACT_DIR, "scaler.joblib"))

feature_cols = joblib.load(os.path.join(ARTIFACT_DIR, "feature_cols.joblib"))
target_cols = joblib.load(os.path.join(ARTIFACT_DIR, "target_cols.joblib"))

xgb_models = joblib.load(os.path.join(ARTIFACT_DIR, "xgb_models.joblib"))
blend_weights = joblib.load(os.path.join(ARTIFACT_DIR, "blend_weights.joblib"))

with open(os.path.join(ARTIFACT_DIR, "keras_metadata.json"), "r", encoding="utf-8") as f:
    keras_metadata = json.load(f)

loaded_keras_items = []

for meta in keras_metadata:
    model_path = os.path.join(ARTIFACT_DIR, meta["model_path"])

    model = keras.models.load_model(model_path)

    loaded_keras_items.append({
        "model": model,
        "targets": meta["targets"],
        "use_log_targets": meta["use_log_targets"],
        "y_mean": meta["y_mean"],
        "y_std": meta["y_std"],
        "kind": meta["kind"],
        "seed": meta["seed"],
    })

print("Модель загружена")
print("Targets:", target_cols)
print("Feature count:", len(feature_cols))

Модель загружена
Targets: ['agl_debye', 'agl_heat_capacity_Cv_300K', 'agl_heat_capacity_Cp_300K', 'agl_thermal_conductivity_300K', 'agl_thermal_expansion_300K', 'agl_bulk_modulus_isothermal_300K', 'agl_bulk_modulus_static_300K', 'agl_acoustic_debye', 'agl_gruneisen']
Feature count: 127


In [3]:
import pandas as pd
import numpy as np

def input_float(prompt, allow_empty=True):
    value = input(prompt).strip().replace(",", ".")

    if value == "" and allow_empty:
        return np.nan

    try:
        return float(value)
    except ValueError:
        print("Ошибка: нужно ввести число.")
        return input_float(prompt, allow_empty=allow_empty)


def input_int(prompt, allow_empty=False):
    value = input(prompt).strip()

    if value == "" and allow_empty:
        return np.nan

    try:
        return int(value)
    except ValueError:
        print("Ошибка: нужно ввести целое число.")
        return input_int(prompt, allow_empty=allow_empty)


# =========================
# ВВОД ОДНОЙ СТРОКИ
# =========================

row = {
    "compound": input("compound, например Te2Zn2: ").strip(),

    "volume_atom": input_float("volume_atom: "),
    "density": input_float("density: "),
    "energy_atom": input_float("energy_atom: "),

    # можно оставить пустым, если неизвестно
    "enthalpy_formation_atom": input_float(
        "enthalpy_formation_atom, если неизвестно — Enter: ",
        allow_empty=True
    ),

    "Egap": input_float("Egap: "),
    "Egap_type": input("Egap_type, например metal / insulator / semiconductor: ").strip(),

    "natoms": input_int("natoms: "),
    "nspecies": input_int("nspecies: "),
    "spacegroup_relax": input_int("spacegroup_relax: "),
}

df_new = pd.DataFrame([row])

# =========================
# СОХРАНЕНИЕ
# =========================

df_new.to_csv("new_material_for_prediction.csv", index=False)

display(df_new)

print("Сохранено в new_material_for_prediction.csv")

compound, например Te2Zn2:  Ac1H2
volume_atom:  17.0642
density:  7.42958
energy_atom:  -4.18878
enthalpy_formation_atom, если неизвестно — Enter:  -0.566575
Egap:  0.0
Egap_type, например metal / insulator / semiconductor:  metal
natoms:  3
nspecies:  2
spacegroup_relax:  225


,compound,volume_atom,density,energy_atom,enthalpy_formation_atom,Egap,Egap_type,natoms,nspecies,spacegroup_relax
0,Ac1H2,17.0642,7.42958,-4.18878,-0.566575,0.0,metal,3,2,225


Сохранено в new_material_for_prediction.csv


In [2]:
import os
import re
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from mendeleev import element

warnings.filterwarnings(
    "ignore",
    message=".*has multiple allotropes.*",
    category=UserWarning
)

# =========================
# 1. LOAD SAVED MODEL ARTIFACTS
# =========================

ARTIFACT_DIR = "model"

imputer = joblib.load(os.path.join(ARTIFACT_DIR, "imputer.joblib"))
scaler = joblib.load(os.path.join(ARTIFACT_DIR, "scaler.joblib"))

feature_cols = joblib.load(os.path.join(ARTIFACT_DIR, "feature_cols.joblib"))
target_cols = joblib.load(os.path.join(ARTIFACT_DIR, "target_cols.joblib"))

xgb_models = joblib.load(os.path.join(ARTIFACT_DIR, "xgb_models.joblib"))
blend_weights = joblib.load(os.path.join(ARTIFACT_DIR, "blend_weights.joblib"))

with open(os.path.join(ARTIFACT_DIR, "keras_metadata.json"), "r", encoding="utf-8") as f:
    keras_metadata = json.load(f)

loaded_keras_items = []

for meta in keras_metadata:
    model = keras.models.load_model(
        os.path.join(ARTIFACT_DIR, meta["model_path"]),
        compile=False
    )

    loaded_keras_items.append({
        "model": model,
        "targets": meta["targets"],
        "use_log_targets": meta["use_log_targets"],
        "y_mean": meta["y_mean"],
        "y_std": meta["y_std"],
        "kind": meta["kind"],
        "seed": meta["seed"],
    })

print("Модель загружена")
print("Targets:", target_cols)
print("Feature count:", len(feature_cols))


# =========================
# 2. LOAD NEW MATERIAL
# =========================

df_new = pd.read_csv("new_material_for_prediction.csv")

display(df_new)


# =========================
# 3. FEATURE ENGINEERING FOR NEW DATA
# =========================

def parse_formula(formula):
    if pd.isna(formula):
        return {}

    formula = str(formula).strip().replace(" ", "")
    parts = re.findall(r"([A-Z][a-z]?)([0-9]*\.?[0-9]*)", formula)

    comp = {}

    for el, amount in parts:
        amount = float(amount) if amount != "" else 1.0
        comp[el] = comp.get(el, 0.0) + amount

    return comp


_element_cache = {}

def safe_float(x):
    try:
        if x is None:
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def get_element_props(symbol):
    if symbol in _element_cache:
        return _element_cache[symbol]

    try:
        el = element(symbol)

        props = {
            "Z": safe_float(getattr(el, "atomic_number", np.nan)),
            "atomic_weight": safe_float(getattr(el, "atomic_weight", np.nan)),
            "period": safe_float(getattr(el, "period", np.nan)),
            "en_pauling": safe_float(getattr(el, "en_pauling", np.nan)),
            "atomic_radius": safe_float(getattr(el, "atomic_radius", np.nan)),
            "covalent_radius": safe_float(getattr(el, "covalent_radius_pyykko", np.nan)),
            "melting_point": safe_float(getattr(el, "melting_point", np.nan)),
            "specific_heat": safe_float(getattr(el, "specific_heat", np.nan)),
        }

        try:
            props["ionization_energy"] = safe_float(el.ionenergies.get(1, np.nan))
        except Exception:
            props["ionization_energy"] = np.nan

    except Exception as e:
        print("MENDELEEV ERROR:", symbol, repr(e))

        props = {
            "Z": np.nan,
            "atomic_weight": np.nan,
            "period": np.nan,
            "en_pauling": np.nan,
            "atomic_radius": np.nan,
            "covalent_radius": np.nan,
            "melting_point": np.nan,
            "specific_heat": np.nan,
            "ionization_energy": np.nan,
        }

    _element_cache[symbol] = props
    return props


def weighted_avg_abs_dev(vals, weights, mean):
    return np.sum(weights * np.abs(vals - mean))


def composition_features(formula):
    comp = parse_formula(formula)

    if len(comp) == 0:
        return {}

    elements = list(comp.keys())
    amounts = np.array(list(comp.values()), dtype=float)

    total_atoms = amounts.sum()
    fractions = amounts / (total_atoms + 1e-12)

    feats = {}

    feats["comp_n_elements"] = len(elements)
    feats["comp_total_atoms"] = total_atoms
    feats["comp_max_fraction"] = fractions.max()
    feats["comp_fraction_l2"] = np.sqrt(np.sum(fractions ** 2))
    feats["comp_config_entropy"] = -np.sum(fractions * np.log(fractions + 1e-12))

    prop_names = [
        "Z",
        "atomic_weight",
        "period",
        "en_pauling",
        "atomic_radius",
        "covalent_radius",
        "melting_point",
        "specific_heat",
        "ionization_energy",
    ]

    prop_values = {p: [] for p in prop_names}

    for el_symbol in elements:
        props = get_element_props(el_symbol)

        for p in prop_names:
            prop_values[p].append(props[p])

    for p in prop_names:
        vals = np.array(prop_values[p], dtype=float)
        valid = ~np.isnan(vals)

        if valid.sum() == 0:
            feats[f"{p}_mean_w"] = np.nan
            feats[f"{p}_min"] = np.nan
            feats[f"{p}_max"] = np.nan
            feats[f"{p}_range"] = np.nan
            feats[f"{p}_std_w"] = np.nan
            feats[f"{p}_avg_abs_dev_w"] = np.nan
            continue

        vals_valid = vals[valid]
        fr_valid = fractions[valid]
        fr_valid = fr_valid / (fr_valid.sum() + 1e-12)

        mean_w = np.sum(vals_valid * fr_valid)
        var_w = np.sum(fr_valid * (vals_valid - mean_w) ** 2)

        feats[f"{p}_mean_w"] = mean_w
        feats[f"{p}_min"] = vals_valid.min()
        feats[f"{p}_max"] = vals_valid.max()
        feats[f"{p}_range"] = vals_valid.max() - vals_valid.min()
        feats[f"{p}_std_w"] = np.sqrt(var_w)
        feats[f"{p}_avg_abs_dev_w"] = weighted_avg_abs_dev(vals_valid, fr_valid, mean_w)

    return feats


def make_features_for_prediction(df_raw):
    df_fe = df_raw.copy()

    numeric_base_cols = [
        "volume_atom",
        "volume_cell",
        "density",
        "energy_atom",
        "enthalpy_formation_atom",
        "Egap",
        "natoms",
        "nspecies",
        "spacegroup_relax",
    ]

    for col in numeric_base_cols:
        if col in df_fe.columns:
            df_fe[col] = pd.to_numeric(df_fe[col], errors="coerce")

    comp_feat_df = pd.DataFrame(
        df_fe["compound"].apply(composition_features).tolist(),
        index=df_fe.index
    )

    df_fe = pd.concat([df_fe, comp_feat_df], axis=1)

    all_elements = set()

    for formula in df_fe["compound"]:
        all_elements.update(parse_formula(formula).keys())

    all_elements = sorted(all_elements)

    element_fraction_rows = []

    for formula in df_fe["compound"]:
        comp = parse_formula(formula)
        row = {f"el_{el}": 0.0 for el in all_elements}

        if len(comp) > 0:
            total = sum(comp.values())

            for el, amount in comp.items():
                row[f"el_{el}"] = amount / (total + 1e-12)

        element_fraction_rows.append(row)

    element_fraction_df = pd.DataFrame(element_fraction_rows, index=df_fe.index)
    df_fe = pd.concat([df_fe, element_fraction_df], axis=1)

    eps = 1e-9

    def add_if_cols_exist(new_col, cols, func):
        if set(cols).issubset(df_fe.columns):
            try:
                df_fe[new_col] = func(df_fe)
            except Exception:
                df_fe[new_col] = np.nan

    add_if_cols_exist("inv_volume_atom", ["volume_atom"], lambda d: 1.0 / (d["volume_atom"] + eps))
    add_if_cols_exist("log_volume_atom", ["volume_atom"], lambda d: np.log1p(d["volume_atom"].clip(lower=0)))
    add_if_cols_exist("volume_atom_pow_minus_1_3", ["volume_atom"], lambda d: 1.0 / np.power(d["volume_atom"] + eps, 1.0 / 3.0))
    add_if_cols_exist("volume_atom_pow_minus_2_3", ["volume_atom"], lambda d: 1.0 / np.power(d["volume_atom"] + eps, 2.0 / 3.0))
    add_if_cols_exist("inv_density", ["density"], lambda d: 1.0 / (d["density"] + eps))

    add_if_cols_exist("abs_energy_atom", ["energy_atom"], lambda d: d["energy_atom"].abs())
    add_if_cols_exist("energy_density_proxy", ["energy_atom", "volume_atom"], lambda d: d["energy_atom"].abs() / (d["volume_atom"] + eps))
    add_if_cols_exist("energy_density_signed", ["energy_atom", "volume_atom"], lambda d: d["energy_atom"] / (d["volume_atom"] + eps))

    add_if_cols_exist("density_div_atomic_weight", ["density", "atomic_weight_mean_w"], lambda d: d["density"] / (d["atomic_weight_mean_w"] + eps))
    add_if_cols_exist("Z_div_atomic_weight", ["Z_mean_w", "atomic_weight_mean_w"], lambda d: d["Z_mean_w"] / (d["atomic_weight_mean_w"] + eps))
    add_if_cols_exist("radius_div_volume_atom", ["atomic_radius_mean_w", "volume_atom"], lambda d: d["atomic_radius_mean_w"] / (d["volume_atom"] + eps))
    add_if_cols_exist("volume_div_radius3", ["volume_atom", "atomic_radius_mean_w"], lambda d: d["volume_atom"] / (np.power(d["atomic_radius_mean_w"], 3) + eps))

    add_if_cols_exist("melting_div_atomic_weight", ["melting_point_mean_w", "atomic_weight_mean_w"], lambda d: d["melting_point_mean_w"] / (d["atomic_weight_mean_w"] + eps))

    if "Egap" in df_fe.columns:
        df_fe["Egap"] = pd.to_numeric(df_fe["Egap"], errors="coerce")
        df_fe["log_Egap"] = np.log1p(df_fe["Egap"].clip(lower=0))
        df_fe["Egap_is_zero"] = (df_fe["Egap"].fillna(0) <= 1e-6).astype(float)

    if "Egap_type" in df_fe.columns:
        egap_type = df_fe["Egap_type"].astype(str).str.lower()
        df_fe["Egap_type_is_metal"] = (egap_type == "metal").astype(float)

    if "spacegroup_relax" in df_fe.columns:
        df_fe["spacegroup_relax"] = pd.to_numeric(df_fe["spacegroup_relax"], errors="coerce")

    df_fe["sound_velocity_proxy"] = np.sqrt(
        df_fe["energy_density_proxy"].clip(lower=0) / (df_fe["density"] + eps)
    )

    df_fe["debye_proxy"] = (
        df_fe["sound_velocity_proxy"] *
        df_fe["volume_atom_pow_minus_1_3"]
    )

    df_fe["atomic_weight_rel_range"] = (
        df_fe["atomic_weight_range"] /
        (df_fe["atomic_weight_mean_w"] + eps)
    )

    df_fe["atomic_radius_rel_range"] = (
        df_fe["atomic_radius_range"] /
        (df_fe["atomic_radius_mean_w"] + eps)
    )

    df_fe["en_radius_interaction"] = (
        df_fe["en_pauling_mean_w"] *
        df_fe["atomic_radius_mean_w"]
    )

    df_fe["en_Z_interaction"] = (
        df_fe["en_pauling_mean_w"] *
        df_fe["Z_mean_w"]
    )

    light_elements = ["H", "B", "C", "N", "O"]
    light_cols = [f"el_{el}" for el in light_elements if f"el_{el}" in df_fe.columns]

    if len(light_cols) > 0:
        df_fe["light_element_fraction"] = df_fe[light_cols].sum(axis=1)
    else:
        df_fe["light_element_fraction"] = 0.0

    df_fe = df_fe.replace([np.inf, -np.inf], np.nan)

    return df_fe


df_fe_new = make_features_for_prediction(df_new)

X_new = df_fe_new.reindex(columns=feature_cols)
X_new = X_new.select_dtypes(include=[np.number])

X_new_imp = imputer.transform(X_new)
X_new_scaled = scaler.transform(X_new_imp)


# =========================
# 4. PREDICT KERAS
# =========================

def pred_list_to_matrix(pred_list):
    if isinstance(pred_list, list):
        return np.column_stack([p.flatten() for p in pred_list])

    pred_arr = np.asarray(pred_list)

    if pred_arr.ndim == 1:
        pred_arr = pred_arr.reshape(-1, 1)

    return pred_arr


positive_targets = {
    "agl_debye",
    "agl_acoustic_debye",
    "agl_heat_capacity_Cv_300K",
    "agl_heat_capacity_Cp_300K",
    "agl_thermal_conductivity_300K",
    "agl_bulk_modulus_isothermal_300K",
    "agl_bulk_modulus_static_300K",
}

keras_pred_lists = {target: [] for target in target_cols}

for item in loaded_keras_items:
    model = item["model"]
    targets = item["targets"]
    use_log_targets = set(item["use_log_targets"])

    y_mean = pd.Series(item["y_mean"])
    y_std = pd.Series(item["y_std"])

    pred_scaled = pred_list_to_matrix(model.predict(X_new_scaled, verbose=0))

    pred_work = pred_scaled * y_std[targets].values + y_mean[targets].values
    pred_real = pred_work.copy()

    for j, target in enumerate(targets):
        if target in use_log_targets:
            pred_real[:, j] = np.expm1(pred_real[:, j])

        if target in positive_targets:
            pred_real[:, j] = np.clip(pred_real[:, j], 0, None)

        keras_pred_lists[target].append(pred_real[:, j])

keras_pred_by_target = {}

for target in target_cols:
    keras_pred_by_target[target] = np.mean(keras_pred_lists[target], axis=0)


# =========================
# 5. PREDICT XGBOOST
# =========================

xgb_pred_by_target = {}

for target in target_cols:
    item = xgb_models[target]

    model = item["model"]
    use_log = item["use_log"]

    pred = model.predict(X_new_imp)

    if use_log:
        pred = np.expm1(pred)

    if target in positive_targets:
        pred = np.clip(pred, 0, None)

    xgb_pred_by_target[target] = pred


# =========================
# 6. FINAL BLEND
# =========================

final_pred_by_target = {}

for target in target_cols:
    w_keras = blend_weights[target]["keras_weight"]
    w_xgb = blend_weights[target]["xgb_weight"]

    final_pred_by_target[target] = (
        w_keras * keras_pred_by_target[target] +
        w_xgb * xgb_pred_by_target[target]
    )


# =========================
# 7. RESULT
# =========================

prediction_result = pd.DataFrame()

if "compound" in df_new.columns:
    prediction_result["compound"] = df_new["compound"].values

for target in target_cols:
    prediction_result[f"pred_{target}"] = final_pred_by_target[target]

display(prediction_result)

prediction_result.to_csv("prediction_result.csv", index=False)

print("Сохранено в prediction_result.csv")

Модель загружена
Targets: ['agl_debye', 'agl_heat_capacity_Cv_300K', 'agl_heat_capacity_Cp_300K', 'agl_thermal_conductivity_300K', 'agl_thermal_expansion_300K', 'agl_bulk_modulus_isothermal_300K', 'agl_bulk_modulus_static_300K', 'agl_acoustic_debye', 'agl_gruneisen']
Feature count: 127


,compound,volume_atom,density,energy_atom,enthalpy_formation_atom,Egap,Egap_type,natoms,nspecies,spacegroup_relax
0,Ac1H2,17.0642,7.42958,-4.18878,-0.566575,0.0,metal,3,2,225


C:\Program Files\Python39\lib\site-packages\xgboost\core.py:158: UserWarning: [18:07:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


,compound,pred_agl_debye,pred_agl_heat_capacity_Cv_300K,pred_agl_heat_capacity_Cp_300K,pred_agl_thermal_conductivity_300K,pred_agl_thermal_expansion_300K,pred_agl_bulk_modulus_isothermal_300K,pred_agl_bulk_modulus_static_300K,pred_agl_acoustic_debye,pred_agl_gruneisen
0,Ac1H2,244.775095,8.343796,8.805741,6.969581,0.000081,41.726628,49.682714,179.766387,1.510062


Сохранено в prediction_result.csv
